# 📈 Retail Demand Forecasting Pipeline — Annotated Study Version
**Author:** Bawelile Gule · Data Scientist & AI Strategist  
**Stack:** Python · Prophet · XGBoost · MLflow · Plotly  
[LinkedIn](https://www.linkedin.com/in/BawelileGule1010) · [Portfolio](https://Bawelile-Gule.github.io)

---

## 📚 How to use this notebook
This is the **annotated study version** of the pipeline. Every cell includes:
- `# WHY:` — the business or statistical reason for this decision
- `# HOW:` — what the code is actually doing, step by step
- `# INTERVIEW TIP:` — how to explain this in a technical interview
- `# WATCH OUT:` — common mistakes candidates make

Read every comment. They are the difference between *running code* and *understanding it*.

---

## Business Problem

Retail operations live and die by demand forecasting accuracy:
- **Over-forecast** → you make or order too much → wasted inventory, spoilage costs, cash tied up in stock
- **Under-forecast** → you don't make enough → lost sales, empty shelves, disappointed customers

The goal of this pipeline is to predict **how many units of each product each store will sell each day**, far enough in advance that operations teams can act on it.

---

## Full Pipeline Architecture
```
STAGE 1: Generate / Load Raw Data
         ↓
STAGE 2: Ingest & Validate  ← schema checks, missing value imputation, sanity checks
         ↓
STAGE 3: Exploratory Analysis  ← understand patterns before modelling
         ↓
STAGE 4: Feature Engineering  ← transform raw dates into signals the model can learn from
         ↓
STAGE 5: Temporal Train/Test Split  ← NEVER shuffle time-series data
         ↓
STAGE 6: Train Models  ← Prophet (seasonality) + XGBoost (lag features) + Ensemble
         ↓  (all logged to MLflow)
STAGE 7: Evaluate  ← RMSE, MAPE → translate to business dollar impact
         ↓
STAGE 8: Forward Forecast  ← generate future predictions with uncertainty bounds
```

> **▶ Run all cells top to bottom.** The full pipeline takes ~2–3 minutes on Colab.

---
## Stage 0 · Install dependencies

**WHY these libraries?**
- `prophet` — Meta's open-source time-series forecasting library. Handles trend, seasonality, and holidays automatically without manual feature engineering.
- `mlflow` — Industry-standard experiment tracking tool. Every model training run gets logged with its parameters and metrics so you can compare runs and reproduce results.
- `xgboost` — Gradient boosting library. Extremely powerful for tabular data with engineered features. Used by winning teams in almost every Kaggle forecasting competition.
- `plotly` — Interactive charting library. Unlike matplotlib (static), Plotly charts are zoomable and hoverable — much better for dashboards and portfolio demos.

In [ ]:
# %%capture suppresses the installation output so it doesn't clutter the notebook
# --quiet reduces pip's verbosity
# These are not pre-installed on Colab's default Python environment
%%capture
!pip install prophet mlflow xgboost plotly --quiet

---
## Stage 1 · Imports & configuration

**WHY centralise config in a dictionary?**  
In production ML systems, you never hardcode values like store names, test window sizes, or price assumptions directly inside functions. Centralising them in a `CONFIG` dict means:
1. You change one place, everything updates
2. You can swap configs per environment (dev vs prod)
3. MLflow can log the entire config as a parameter set for reproducibility

**INTERVIEW TIP:** If asked *'how would you make this pipeline configurable?'* — this is the answer. Talk about config dicts, YAML files, or environment variables depending on the scale.

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress noisy Prophet/MLflow warnings during runs

# Core data libraries
import numpy as np          # numerical operations, array math
import pandas as pd         # DataFrame operations, time-series indexing

# Modelling libraries
import xgboost as xgb       # gradient boosted trees
from prophet import Prophet  # Meta's time-series model
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Experiment tracking
import mlflow               # logs every training run automatically

# Visualisation
import plotly.graph_objects as go  # low-level Plotly (full control)
import plotly.express as px        # high-level Plotly (quick charts)
from plotly.subplots import make_subplots  # side-by-side charts

from IPython.display import display  # renders DataFrames nicely in Colab

# ── Set random seed for reproducibility ─────────────────────────────────────
# WHY: np.random.seed ensures that any random operations (noise generation,
# XGBoost subsampling) produce the same result every run. Without this,
# results change each time you run the notebook — bad for reproducibility.
np.random.seed(42)

# ── Centralised pipeline configuration ──────────────────────────────────────
# HOW: Change these values to run the full pipeline on a different store/product
# without touching any other code.
CONFIG = {
    'store'      : 'ATL-01',           # Which store to analyse
    'product'    : 'Cinnamon Classic', # Which product to forecast
    'test_days'  : 60,                 # How many days to hold out for evaluation
    'horizon'    : 30,                 # How many days ahead to forecast
    'avg_price'  : 5.75,              # Average unit price (for $ impact calculation)
}

# Available stores and products in this dataset
STORES   = ['ATL-01', 'ATL-02', 'ATL-03', 'CHI-01', 'NYC-01']
PRODUCTS = ['Cinnamon Classic', 'Vegan Delight', 'Choco Fudge', 'Berry Bliss', 'Nutella Dream']

print('✅ Libraries loaded successfully.')
print(f"\nActive configuration:")
for k, v in CONFIG.items():
    print(f"   {k:<20}: {v}")

---
## Stage 2 · Generate synthetic retail dataset

**WHY synthetic data?**  
Real retail data is proprietary. Synthetic data lets us build and demonstrate a production-quality pipeline without violating any NDAs. The key is making the synthetic data *realistic* — it needs to exhibit the same statistical properties as real retail data.

**What makes retail demand realistic?**  
Real demand has several overlapping signals:
1. **Trend** — gradual growth or decline over months/years (new store opens, competition arrives)
2. **Yearly seasonality** — predictable annual patterns (holiday peaks, summer slumps)
3. **Weekly seasonality** — consistent day-of-week patterns (weekends are busier for food)
4. **Promotional lift** — temporary spikes during sales events (18–28% uplift)
5. **Holiday spikes** — large short-term spikes on holidays
6. **Random noise** — unpredictable variation (weather, local events, competitor promotions)
7. **Missing data** — real systems always have gaps (server outages, data pipeline failures)

**INTERVIEW TIP:** If asked *'why does your dataset have missing values?'* — explain that ~2% missing is realistic and that your ingestion pipeline handles it explicitly. This shows you've thought about production conditions, not just clean toy datasets.

In [ ]:
# ── Constants that define the data generation process ───────────────────────

# Weeks of the year that have promotional events (week numbers 1-52)
# These correspond to: New Year, Easter, Memorial Day, 4th of July,
# Back-to-School, Halloween, and Christmas periods
PROMO_WEEKS = [4, 13, 22, 27, 35, 44, 50]

# Specific holiday dates — these cause large, short-duration spikes
HOLIDAYS = pd.to_datetime([
    '2022-07-04', '2022-11-24', '2022-12-25',   # 2022 holidays
    '2023-07-04', '2023-11-23', '2023-12-25',   # 2023 holidays
])

# Per-product configuration
# 'base'  = the baseline average daily demand (units)
# 'amp'   = seasonal amplitude (how much demand swings with the season)
# 'price' = average unit price in dollars
# NOTE: Different products have different demand levels and seasonal sensitivity
PRODUCT_CFG = {
    'Cinnamon Classic': {'base': 120, 'amp': 0.25, 'price': 5.50},
    'Vegan Delight'   : {'base':  85, 'amp': 0.15, 'price': 6.25},
    'Choco Fudge'     : {'base': 100, 'amp': 0.30, 'price': 5.75},
    'Berry Bliss'     : {'base':  70, 'amp': 0.40, 'price': 5.95},
    'Nutella Dream'   : {'base':  95, 'amp': 0.20, 'price': 6.50},
}


def generate_series(start, end, product, store):
    """
    Generate a synthetic daily demand time series for one store-product pair.

    HOW it works — we build demand as a SUM of components:
        demand = trend + yearly_seasonality + weekly_seasonality
                 + promo_lift + holiday_lift + weekend_lift + noise

    This additive structure mirrors how time-series decomposition works
    (which we'll use in the EDA stage to verify our model captures each component).
    """
    cfg   = PRODUCT_CFG[product]
    dates = pd.date_range(start, end, freq='D')  # one row per day
    n     = len(dates)
    t     = np.arange(n)                          # integer time index: 0, 1, 2, ..., n

    # COMPONENT 1: Trend
    # WHY: Real demand grows slightly over time (brand awareness, store maturity)
    # HOW: Linear trend starting at cfg['base'], growing at 0.03 units/day
    trend = cfg['base'] + 0.03 * t

    # COMPONENT 2: Yearly seasonality
    # WHY: Demand follows an annual cycle (e.g. pastry demand peaks in winter)
    # HOW: A sine wave with period 365 days. sin(2π * t / 365) completes one
    # full cycle per year. The amplitude scales with the product's seasonal sensitivity.
    yearly = cfg['amp'] * cfg['base'] * np.sin(2 * np.pi * t / 365)

    # COMPONENT 3: Weekly seasonality
    # WHY: Weekends are busier for food retail
    # HOW: A sine wave with period 7 days — one cycle per week
    # Amplitude fixed at 12% of base demand
    weekly = 0.12 * cfg['base'] * np.sin(2 * np.pi * t / 7)

    # COMPONENT 4: Random noise
    # WHY: Real demand is never perfectly predictable — weather, local events,
    # competitor promotions all cause unexplained variation
    # HOW: Gaussian noise with std = 8% of base demand
    noise  = np.random.normal(0, cfg['base'] * 0.08, n)

    # Build the initial DataFrame
    df = pd.DataFrame({'date': dates})
    df['store']   = store
    df['product'] = product

    # Extract week number for promo matching
    df['week'] = df['date'].dt.isocalendar().week.astype(int)

    # Binary flags for external signals
    # WHY: The model needs to know *when* special events occur to learn their effect
    df['is_promo']   = df['week'].isin(PROMO_WEEKS).astype(int)        # 1 on promo weeks
    df['is_holiday'] = df['date'].isin(HOLIDAYS).astype(int)           # 1 on holidays
    df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)      # 1 on Sat/Sun

    # COMPONENT 5: Promotional lift
    # WHY: Promos drive 18–30% additional sales. We use np.random.uniform to
    # vary the lift — not all promotions are equally effective
    promo_lift   = df['is_promo']   * cfg['base'] * np.random.uniform(0.18, 0.30)

    # COMPONENT 6: Holiday lift
    # WHY: Holidays drive 40–60% spikes — people buy treats for gatherings
    holiday_lift = df['is_holiday'] * cfg['base'] * np.random.uniform(0.40, 0.60)

    # COMPONENT 7: Weekend lift
    # WHY: Fixed 15% uplift on weekends — foot traffic is consistently higher
    weekend_lift = df['is_weekend'] * cfg['base'] * 0.15

    # Add all components together
    raw_demand = trend + yearly + weekly + promo_lift + holiday_lift + weekend_lift + noise

    # np.clip(0) ensures no negative demand (can't sell -5 units)
    # .round() converts to whole units (you can't sell 0.7 of a cinnamon roll)
    df['units_sold'] = np.clip(raw_demand, 0, None).round().astype(int)

    # Simulate inventory levels
    # WHY: Inventory is a related signal — if inventory drops below demand, we
    # might be stock-limited (understated demand). In a production model, we'd
    # correct for this censored demand.
    df['inventory_on_hand'] = (
        df['units_sold'] * np.random.uniform(1.1, 2.5, n)
    ).round().astype(int)

    # Calculate revenue
    df['revenue'] = (df['units_sold'] * cfg['price']).round(2)

    # INTENTIONALLY introduce ~2% missing values
    # WHY: Real data pipelines have outages, sensor failures, and human errors.
    # A model trained only on clean data will fail in production. We simulate
    # this and handle it explicitly in Stage 3.
    for col in ['units_sold', 'inventory_on_hand', 'revenue']:
        mask = np.random.random(n) < 0.02   # 2% probability each row
        df.loc[mask, col] = np.nan          # replace with NaN

    return df


# ── Generate the full dataset (5 stores × 5 products × 2 years) ─────────────
frames = []
for store in STORES:
    for product in PRODUCTS:
        frames.append(generate_series('2022-01-01', '2023-12-31', product, store))

# pd.concat stacks all frames vertically into one big DataFrame
RAW_DF = pd.concat(frames, ignore_index=True)
RAW_DF = RAW_DF.sort_values(['store', 'product', 'date']).reset_index(drop=True)

print(f'✅ Dataset generated successfully')
print(f'   Rows        : {len(RAW_DF):,}  ({len(STORES)} stores × {len(PRODUCTS)} products × ~730 days)')
print(f'   Date range  : {RAW_DF["date"].min().date()} → {RAW_DF["date"].max().date()}')
print(f'   Columns     : {RAW_DF.columns.tolist()}')
print(f'   Missing vals: {RAW_DF.isnull().sum().sum()} (~2% intentional)')
print()
RAW_DF.head()

---
## Stage 3 · Ingest & validate

**WHY have a separate ingestion stage?**  
In real pipelines, data comes from databases, APIs, or flat files — and it's almost never clean. The ingestion stage is the **quality gate**: nothing enters the modelling pipeline without passing checks. This is called a *data contract*.

**What does good ingestion include?**
1. **Schema validation** — does the file have all required columns?
2. **Type casting** — are columns the right data types? (dates as dates, not strings)
3. **Missing value reporting** — where are the gaps? How many?
4. **Imputation** — fill gaps in a way that respects the data's structure
5. **Sanity checks** — does the data pass basic logic tests? (no negative sales)

**WHY forward-fill then backward-fill?**  
For time-series data, forward-fill (`ffill`) carries the last known value forward — like assuming *'sales were similar to yesterday'*. This is almost always better than replacing with the global mean, which ignores the local trend and seasonality.

**WATCH OUT:** Do NOT impute across different products or stores. A missing value for *Cinnamon Classic in ATL-01* should be filled from the previous *Cinnamon Classic in ATL-01* observation — not from a different product. That's why we `groupby(['store','product'])` before imputing.

**INTERVIEW TIP:** If asked *'how do you handle missing data in time-series?'* — explain the groupby-ffill approach and why global imputation (mean/median across all groups) destroys the local signal.

In [ ]:
def ingest(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Quality gate: validates, casts types, imputes, and sanity-checks
    the raw DataFrame before it enters the modelling pipeline.
    """

    # ── STEP 1: Schema validation ────────────────────────────────────────────
    # WHY: If a column is missing, every downstream stage will fail with
    # confusing KeyError messages. Better to catch it here with a clear message.
    required_cols = [
        'date', 'store', 'product', 'units_sold',
        'revenue', 'inventory_on_hand',
        'is_promo', 'is_holiday', 'is_weekend'
    ]
    missing_cols = [c for c in required_cols if c not in raw.columns]
    assert not missing_cols, f'Missing required columns: {missing_cols}'
    print(f'✅ Schema OK — all {len(required_cols)} required columns present')

    df = raw.copy()  # always work on a copy — never mutate the original

    # ── STEP 2: Type casting ─────────────────────────────────────────────────
    # WHY: 'category' dtype uses less memory than 'object' (string) and makes
    # groupby operations faster. With 18,250 rows, this matters less — but at
    # millions of rows, it's critical.
    df['store']   = df['store'].astype('category')
    df['product'] = df['product'].astype('category')

    # ── STEP 3: Missing value audit ──────────────────────────────────────────
    # WHY: Before imputing, log exactly what's missing. In production, this
    # would trigger an alert if missingness exceeds a threshold (e.g. >5%).
    null_report = df.isnull().sum()
    null_report = null_report[null_report > 0]  # only show columns WITH nulls

    print('\nMissing values detected (before imputation):')
    if null_report.empty:
        print('   None — data is complete!')
    else:
        for col, count in null_report.items():
            pct = count / len(df) * 100
            print(f'   {col:<25}: {count:>5} rows ({pct:.1f}%)')

    # ── STEP 4: Imputation ───────────────────────────────────────────────────
    # WHY: We impute per-group (store + product) so that ATL-01/Cinnamon Classic
    # missing values are filled from ATL-01/Cinnamon Classic's own history,
    # not from a different product's values.
    #
    # HOW: groupby creates a sub-frame for each store-product pair.
    # transform() applies the function and returns a result with the same index
    # as the original df (so it can be assigned back directly).
    # ffill() = forward fill: carry last valid value forward
    # bfill() = backward fill: if ffill doesn't work (gap at start), go backward
    numeric_cols = ['units_sold', 'inventory_on_hand', 'revenue']
    df[numeric_cols] = (
        df.groupby(['store', 'product'], observed=True)[numeric_cols]
        .transform(lambda s: s.ffill().bfill())
    )

    # Fallback: if an entire series is missing (edge case), fill with column median
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
            print(f'   WARNING: Filled remaining nulls in {col} with global median')

    print('\n✅ Imputation complete — all nulls resolved')

    # ── STEP 5: Sanity checks ────────────────────────────────────────────────
    # WHY: Even after imputation, we assert basic business logic.
    # Negative sales would corrupt the model and produce nonsense forecasts.
    assert (df['units_sold']        >= 0).all(), 'FAILED: Negative units_sold detected'
    assert (df['revenue']           >= 0).all(), 'FAILED: Negative revenue detected'
    assert (df['inventory_on_hand'] >= 0).all(), 'FAILED: Negative inventory detected'
    print('✅ Sanity checks passed — no negative values')

    return df


def get_series(df, store, product):
    """
    Extract the time series for a single store-product pair.

    HOW: Filter → set date as index → reindex to a full date range
    (this fills any missing DATES with NaN, not just missing values within rows)
    → fill sales with 0 for completely missing dates.

    WHY reindex to full date range?
    If a store was closed for a week and no sales were recorded at all
    (not even a row in the data), the time-series would have a DATE GAP.
    Lag features computed across a gap would reference the wrong prior day.
    Reindexing ensures every calendar day has exactly one row.
    """
    s = (
        df[(df['store'] == store) & (df['product'] == product)]
        .copy()
        .set_index('date')
        .sort_index()
    )
    # Create a complete date range with no gaps
    full_idx = pd.date_range(s.index.min(), s.index.max(), freq='D')
    s = s.reindex(full_idx)          # inserts NaN rows for any missing dates
    s['units_sold'] = s['units_sold'].fillna(0)  # closed = 0 sales, not NaN
    s.index.name = 'date'
    return s.reset_index()


# Run ingestion
CLEAN_DF = ingest(RAW_DF)
SERIES   = get_series(CLEAN_DF, CONFIG['store'], CONFIG['product'])

print(f'\nExtracted series: {CONFIG["store"]} | {CONFIG["product"]}')
print(f'Shape: {SERIES.shape}  |  Date range: {SERIES["date"].min().date()} → {SERIES["date"].max().date()}')

---
## Stage 4 · Exploratory data analysis (EDA)

**WHY EDA before modelling?**  
You should NEVER build a model on data you haven't looked at. EDA tells you:
- Whether the patterns you assumed exist actually exist
- Which features will be most useful (saves wasted engineering effort)
- Whether data quality issues survived ingestion
- What kind of model is appropriate (linear? tree-based? sequential?)

**INTERVIEW TIP:** When describing your EDA, don't just say *'I looked at the data'*. Name the specific questions you were trying to answer and what you found. Interviewers want to see *structured thinking*, not just plotting.

In [ ]:
# ── EDA Question 1: Are all stores/products performing similarly? ─────────────
# WHY: If one store sells 10x more than another, a single global model will
# underfit the high-volume store and overfit the low-volume one.
# Result: confirms we need per-store-product models (or at least store/product features).

store_avg   = CLEAN_DF.groupby('store',   observed=True)['units_sold'].mean().sort_values()
product_avg = CLEAN_DF.groupby('product', observed=True)['units_sold'].mean().sort_values()

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Avg daily units sold — by store',
    'Avg daily units sold — by product'
))
fig.add_trace(go.Bar(x=store_avg.values,   y=store_avg.index,   orientation='h',
                     marker_color='#1A5276', name='Store'), row=1, col=1)
fig.add_trace(go.Bar(x=product_avg.values, y=product_avg.index, orientation='h',
                     marker_color='#2E86C1', name='Product'), row=1, col=2)
fig.update_layout(height=300, showlegend=False, title_text='Sales distribution overview')
fig.show()
print('Observation: stores and products have meaningfully different demand levels')
print('→ We will train per-store-product models, not one global model')

In [ ]:
# ── EDA Question 2: What does the raw time series look like? ─────────────────
# WHY: Visualising the full series lets us spot:
#   - Visible trend (line going up/down over time)
#   - Seasonality (repeating waves)
#   - Anomalies (sudden spikes/drops that might be data errors)
#   - Whether promo/holiday markers align with visible spikes (sanity check)

s = SERIES.copy()

fig = go.Figure()

# Main sales line
fig.add_trace(go.Scatter(
    x=s['date'], y=s['units_sold'],
    mode='lines', name='Units sold',
    line=dict(color='#1A5276', width=1.2)
))

# Overlay promo and holiday markers — do the spikes align?
promos   = s[s['is_promo']   == 1]
holidays = s[s['is_holiday'] == 1]

fig.add_trace(go.Scatter(
    x=promos['date'], y=promos['units_sold'],
    mode='markers', name='Promo day',
    marker=dict(color='#F39C12', size=5, symbol='triangle-up')
))
fig.add_trace(go.Scatter(
    x=holidays['date'], y=holidays['units_sold'],
    mode='markers', name='Holiday spike',
    marker=dict(color='#E74C3C', size=9, symbol='star')
))

fig.update_layout(
    title=f'Full historical series — {CONFIG["store"]} | {CONFIG["product"]}',
    xaxis_title='Date', yaxis_title='Units sold',
    height=360, legend=dict(orientation='h', y=1.12),
    plot_bgcolor='white'
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()
print('✅ Check: holiday stars should appear at visible spikes → confirms our flags are correct')

In [ ]:
# ── EDA Question 3: Is there a weekly demand pattern? ────────────────────────
# WHY: If weekends are consistently higher, we need a weekend feature AND
# Prophet's weekly_seasonality=True. If there's NO pattern, adding the feature
# adds noise without signal.
# Result informs: which features to engineer in Stage 5.

s['day_name']   = s['date'].dt.day_name()
s['month_name'] = s['date'].dt.month_name()

dow_order   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

dow_avg   = s.groupby('day_name')['units_sold'].mean().reindex(dow_order)
month_avg = s.groupby('month_name')['units_sold'].mean().reindex(month_order)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Avg demand by day of week',
    'Avg demand by month'
))
fig.add_trace(go.Bar(x=dow_order,   y=dow_avg.values,   marker_color='#2E86C1', name='DoW'),   row=1, col=1)
fig.add_trace(go.Bar(x=month_order, y=month_avg.values, marker_color='#1A5276', name='Month'), row=1, col=2)
fig.update_layout(height=330, showlegend=False, title_text='Seasonality patterns')
fig.show()

# Quantify the weekend uplift — gives you a number to quote in interviews
weekday_avg = dow_avg[['Monday','Tuesday','Wednesday','Thursday','Friday']].mean()
weekend_avg = dow_avg[['Saturday','Sunday']].mean()
uplift_pct  = (weekend_avg / weekday_avg - 1) * 100
print(f'Weekend vs weekday uplift: {uplift_pct:.1f}%')
print('→ Confirms: is_weekend feature and weekly_seasonality in Prophet are justified')

In [ ]:
# ── EDA Question 4: How much do promotions actually lift demand? ──────────────
# WHY: If promos have ZERO lift, the is_promo feature is noise and should be
# dropped. If lift is 25%, it's one of the most important features we have.
# This is also a number you can quote to stakeholders: 'our promos drive X% lift'.

promo_df = (
    CLEAN_DF
    .groupby(['product', 'is_promo'], observed=True)['units_sold']
    .mean()
    .unstack()  # pivot is_promo (0 and 1) into separate columns
    .rename(columns={0: 'Non-promo avg', 1: 'Promo avg'})
)
# Calculate lift: how much % higher is promo demand vs non-promo demand?
promo_df['Lift (%)'] = (
    (promo_df['Promo avg'] - promo_df['Non-promo avg'])
    / promo_df['Non-promo avg'] * 100
).round(1)

print('=== Promotional lift analysis ===')
display(promo_df.sort_values('Lift (%)', ascending=False))

fig = go.Figure(go.Bar(
    y=promo_df.index,
    x=promo_df['Lift (%)'].sort_values(),
    orientation='h',
    marker_color='#E67E22'
))
fig.update_layout(title='Promotional sales lift by product (%)', xaxis_title='Lift (%)', height=280)
fig.show()
print('→ Lifts of 18–28% confirm is_promo is a high-value feature — keep it in the model')

---
## Stage 5 · Feature engineering

**WHY is feature engineering critical for XGBoost?**  
XGBoost is a tree-based model — it cannot see the *order* of rows, and it has no built-in concept of 'yesterday' or 'last week'. We have to explicitly create features that give it that temporal context.

**The 5 feature groups we build:**

| Group | Features | WHY |  
|-------|----------|-----|  
| Calendar | day_of_week, month, quarter, etc. | Captures day-of-week and monthly patterns |
| Lag features | lag_7, lag_14, lag_28 | Tells the model what demand was 1, 2, 4 weeks ago |
| Rolling windows | rolling_mean_7, rolling_std_7, rolling_mean_28 | Captures recent trend and variability |
| Trend | t (integer index) | Captures the long-run growth direction |
| External signals | is_promo, is_holiday, is_weekend | Known future events that shift demand |

**WHY lag_7 and not lag_1?**  
For daily retail data, lag_7 (same day last week) is almost always a stronger predictor than lag_1 (yesterday). Monday demand is most similar to last Monday, not Sunday. We include lag_14 and lag_28 to capture two-week and four-week patterns.

**WATCH OUT — data leakage:**  
Rolling features MUST use `.shift(1)` before rolling. If you compute a 7-day rolling mean without shifting, the mean includes today's value — the model would 'see the future' during training. This is called **data leakage** and is one of the most common mistakes in ML.

**INTERVIEW TIP:** When asked about feature engineering for time-series, the three things interviewers want to hear are: (1) lag features, (2) rolling statistics, and (3) leakage prevention via `.shift()`.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform a raw time-series DataFrame into a rich feature matrix
    ready for XGBoost training.

    IMPORTANT: Input must be a single store-product series, date-sorted.
    Mixing multiple products/stores before computing lags would create
    incorrect lag values at the boundaries between groups.
    """
    df = df.copy().sort_values('date').reset_index(drop=True)

    # ── GROUP 1: Calendar features ───────────────────────────────────────────
    # HOW: pandas datetime accessor (.dt) extracts these directly from the date column
    # WHY each one:
    #   day_of_week  → 0=Monday, 6=Sunday. Captures weekday/weekend patterns.
    #   day_of_month → captures pay-day effects (demand up on the 1st and 15th)
    #   week_of_year → captures specific week patterns (Black Friday = week 47)
    #   month        → captures monthly seasonality
    #   quarter      → coarser seasonal signal
    #   is_month_end → some products spike at month end (gift purchases)
    df['day_of_week']    = df['date'].dt.dayofweek
    df['day_of_month']   = df['date'].dt.day
    df['week_of_year']   = df['date'].dt.isocalendar().week.astype(int)
    df['month']          = df['date'].dt.month
    df['quarter']        = df['date'].dt.quarter
    df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)

    # ── GROUP 2: Trend index ─────────────────────────────────────────────────
    # WHY: XGBoost has no inherent sense of 'time passing'. The integer index t
    # gives it a linear trend signal — row 0 is older, row 730 is newer.
    # Without this, the model can't distinguish 'Jan 2022' from 'Jan 2023'.
    df['t'] = np.arange(len(df))

    # ── GROUP 3: Lag features ────────────────────────────────────────────────
    # HOW: .shift(lag) moves the column DOWN by `lag` rows, so that on any
    # given row, lag_7 contains the value from 7 rows (days) earlier.
    # WHY lag_7: same-day-last-week is almost always the strongest predictor
    # WHY lag_14: captures two-week patterns (bi-weekly promotions)
    # WHY lag_28: captures four-week / monthly patterns
    for lag in [7, 14, 28]:
        df[f'lag_{lag}'] = df['units_sold'].shift(lag)

    # ── GROUP 4: Rolling window statistics ───────────────────────────────────
    # CRITICAL: .shift(1) BEFORE .rolling() — this excludes today's value
    # from the window, preventing data leakage.
    #
    # rolling_mean_7  → recent average (last week's trend)
    # rolling_std_7   → recent variability (high std = volatile demand, harder to forecast)
    # rolling_mean_28 → longer-term average (month's trend)
    # rolling_std_28  → longer-term variability
    # rolling_max_7   → highest demand in the last 7 days (catches recent spikes)
    #
    # min_periods=3: compute the stat even if fewer than `window` observations
    # are available (handles the start of the series gracefully)
    for window in [7, 28]:
        df[f'rolling_mean_{window}'] = (
            df['units_sold'].shift(1).rolling(window, min_periods=3).mean()
        )
        df[f'rolling_std_{window}'] = (
            df['units_sold'].shift(1).rolling(window, min_periods=3).std()
        )
    df['rolling_max_7'] = df['units_sold'].shift(1).rolling(7, min_periods=3).max()

    # ── GROUP 5: Price proxy ─────────────────────────────────────────────────
    # WHY: Price is a demand driver. If price went up last month, demand might
    # have dropped. We derive it from revenue / units_sold as a proxy.
    # np.where avoids division by zero when units_sold = 0.
    df['avg_price'] = np.where(
        df['units_sold'] > 0,
        df['revenue'] / df['units_sold'],
        np.nan
    )
    df['avg_price'] = df['avg_price'].ffill().bfill()  # fill any NaN price days

    # Drop the first 28 rows where lag features are NaN
    # WHY: We can't train on rows where the lags are missing — those rows have
    # no lag_7/14/28 because they're too close to the start of the series.
    df = df.dropna(subset=['lag_7', 'lag_14', 'lag_28']).reset_index(drop=True)

    return df


# Ordered list of features the model will actually use
# HOW: Defining this as a list (not deriving from column names) ensures
# the feature order is always consistent between training and inference.
# XGBoost requires the same feature order at prediction time.
FEATURE_COLS = [
    # Calendar
    'day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter',
    'is_month_end', 'is_month_start',
    # Trend
    't',
    # Lags
    'lag_7', 'lag_14', 'lag_28',
    # Rolling
    'rolling_mean_7', 'rolling_std_7',
    'rolling_mean_28', 'rolling_std_28', 'rolling_max_7',
    # Price proxy
    'avg_price',
    # External signals
    'is_promo', 'is_holiday', 'is_weekend',
]

FEAT_DF = build_features(SERIES)
print(f'✅ Feature matrix built: {FEAT_DF.shape[0]} rows × {len(FEATURE_COLS)} features')
print(f'   (Dropped first 28 rows — insufficient lag history)')
print(f'\nFeature preview:')
FEAT_DF[['date'] + FEATURE_COLS[:8]].tail(3)

In [ ]:
# ── Visualise feature correlation with the target ────────────────────────────
# WHY: This confirms which features are actually informative before we train.
# Strong positive correlation → feature helps predict high demand
# Strong negative correlation → feature helps predict low demand
# Near-zero correlation → feature may not be useful (but tree models can still use weak signals)

corr = (
    FEAT_DF[FEATURE_COLS + ['units_sold']]
    .corr()['units_sold']          # correlation of every feature with units_sold
    .drop('units_sold')            # remove the self-correlation (always 1.0)
    .sort_values()                 # sort lowest to highest
)

# Colour code: positive correlations blue, negative red
colors = ['#E74C3C' if v < 0 else '#1A5276' for v in corr.values]

fig = go.Figure(go.Bar(
    x=corr.values, y=corr.index, orientation='h', marker_color=colors
))
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)
fig.update_layout(
    title='Feature correlation with units_sold (Pearson r)',
    xaxis_title='Correlation coefficient', height=520,
    plot_bgcolor='white'
)
fig.show()

top_feature = corr.abs().idxmax()
top_r       = corr.abs().max()
print(f'Strongest predictor: {top_feature} (|r| = {top_r:.3f})')
print(f'\nINTERVIEW NOTE: lag_7 is typically the strongest predictor in weekly retail data.')
print(f'This validates our feature engineering decision to include weekly lags.')

---
## Stage 6 · Train / test split

**WHY temporal split — NEVER random split for time-series:**  
If you randomly shuffle and split time-series data, rows from January 2023 end up in the *training* set while rows from July 2022 end up in the *test* set. This means the model sees *future* data during training — it would learn from information it couldn't have had in real life. This is **temporal leakage** and produces falsely optimistic metrics.

**The correct approach:** always split chronologically. Train on everything up to a cutoff date. Test on everything after. The model is evaluated exactly as it would be in production — predict the future using only the past.

**WHY 60 days as the test window?**  
60 days (2 months) gives enough test data to capture multiple weekend cycles, at least one promo week, and meaningful seasonal variation — without cutting so much that the training set is too small.

**INTERVIEW TIP:** The moment a candidate mentions 'random train-test split' for time-series, most interviewers will immediately follow up with *'isn't that data leakage?'*. Always say *chronological split* for time-series.

In [ ]:
def temporal_split(df, test_days=60):
    """
    Split a time-series DataFrame chronologically.

    HOW:
    1. Find the last date in the dataset
    2. Subtract test_days to get the cutoff date
    3. Train = everything on or before the cutoff
    4. Test  = everything strictly after the cutoff

    WATCH OUT: Using < vs <= on the cutoff date matters.
    With <=, the cutoff date itself is in training (not test).
    This avoids any ambiguity about overlapping dates.
    """
    last_date = df['date'].max()
    cutoff    = last_date - pd.Timedelta(days=test_days)

    train = df[df['date'] <= cutoff].copy()
    test  = df[df['date'] >  cutoff].copy()

    return train, test


TRAIN, TEST = temporal_split(FEAT_DF, test_days=CONFIG['test_days'])

print(f'Train set: {len(TRAIN):>4} rows  ({TRAIN["date"].min().date()} → {TRAIN["date"].max().date()})')
print(f'Test  set: {len(TEST):>4} rows  ({TEST["date"].min().date()} → {TEST["date"].max().date()})')
print(f'Split ratio: {len(TRAIN)/len(FEAT_DF):.0%} train / {len(TEST)/len(FEAT_DF):.0%} test')
print()
print('✅ Confirmed: no chronological overlap between train and test sets')
print(f'   Latest train date ({TRAIN["date"].max().date()}) < Earliest test date ({TEST["date"].min().date()})')

---
## Stage 7 · Model training

**WHY two models instead of one?**  
Each model captures different aspects of demand:

| Model | Strength | Weakness |
|-------|----------|----------|
| **Prophet** | Automatically models trend + yearly + weekly seasonality. Handles holidays natively. Robust to missing data. | Doesn't benefit from lag/rolling features. Can be slow. |
| **XGBoost** | Extremely powerful with engineered features (lags, rolling means). Fast. Handles non-linearity well. | No built-in sense of time — relies entirely on our feature engineering. |
| **Ensemble** | Combines the strengths of both. The errors of one model tend to be offset by the other. | Slightly more complex to maintain. |

**WHY MLflow?**  
MLflow is the industry standard for *experiment tracking*. Without it, you'd have to manually record which parameters produced which metrics in a spreadsheet — error-prone and unscalable. With MLflow, every run is automatically logged and comparable. In interviews, mentioning MLflow signals production-level thinking.

### 7a · Prophet

**HOW Prophet works:**  
Prophet decomposes demand into three additive components:
```
y(t) = trend(t) + seasonality(t) + holidays(t) + error(t)
```
- **Trend:** a piecewise linear or logistic growth curve with automatic changepoint detection
- **Seasonality:** Fourier series decomposition of weekly and yearly patterns
- **Holidays/regressors:** explicit step-function effects for known events

**Key parameters we set:**
- `changepoint_prior_scale=0.05` — controls how flexible the trend is. Lower = smoother, less overfit to short-term fluctuations
- `seasonality_mode='multiplicative'` — seasonal effect scales with the trend level (better for growing series)
- Custom regressors (`is_promo`, `is_holiday`, `is_weekend`) — add our known external signals

In [ ]:
# Set up MLflow — all runs will be logged to this experiment
mlflow.set_experiment('retail-demand-forecasting')

def to_prophet_df(df):
    """
    Prophet requires a DataFrame with exactly two mandatory columns:
      'ds' = date column (datetime)
      'y'  = target column (what we're forecasting)
    Additional columns are treated as extra regressors.
    """
    return (
        df[['date', 'units_sold', 'is_promo', 'is_holiday', 'is_weekend']]
        .rename(columns={'date': 'ds', 'units_sold': 'y'})
        .sort_values('ds')
        .reset_index(drop=True)
    )

p_train = to_prophet_df(TRAIN)
p_test  = to_prophet_df(TEST)

# Prophet hyperparameters
prophet_params = {
    'changepoint_prior_scale': 0.05,        # flexibility of trend — low = smoother
    'seasonality_prior_scale': 10.0,         # flexibility of seasonality — higher = more complex
    'seasonality_mode'       : 'multiplicative',  # use when seasonal amplitude grows with trend
    'yearly_seasonality'     : True,          # model the annual cycle
    'weekly_seasonality'     : True,          # model the day-of-week cycle
}

# mlflow.start_run() creates a new experiment run — everything inside the 'with' block
# gets logged to this run. The run_name lets us find it later.
with mlflow.start_run(run_name=f'prophet_{CONFIG["store"]}_{CONFIG["product"].replace(" ","_")}'):

    # Log all parameters to MLflow so we can reproduce this exact run later
    mlflow.log_params({**prophet_params, 'model': 'prophet',
                       'store': CONFIG['store'], 'product': CONFIG['product']})

    # Build and fit the model
    prophet_model = Prophet(**prophet_params)

    # add_regressor() tells Prophet to include these columns as additional
    # linear effects on top of the base trend+seasonality
    prophet_model.add_regressor('is_promo')
    prophet_model.add_regressor('is_holiday')
    prophet_model.add_regressor('is_weekend')

    prophet_model.fit(p_train)  # train on training data only

    # Predict on the test set
    # HOW: .predict() needs the 'ds' column plus any regressor columns
    # We pass is_promo/is_holiday/is_weekend for the test period so Prophet
    # can apply the correct regressor effects in the forecast
    prophet_forecast = prophet_model.predict(
        p_test[['ds', 'is_promo', 'is_holiday', 'is_weekend']]
    )

    # 'yhat' is Prophet's predicted value
    # .clip(0) ensures no negative predictions (can't forecast -5 units)
    PROPHET_PREDS = prophet_forecast['yhat'].clip(0).values
    ACTUALS       = p_test['y'].values

    # Calculate metrics
    p_rmse = float(np.sqrt(mean_squared_error(ACTUALS, PROPHET_PREDS)))
    p_mape = float(np.mean(np.abs((ACTUALS - PROPHET_PREDS) / (ACTUALS + 1e-8))))
    # NOTE: 1e-8 (a tiny number) is added to avoid division by zero
    # when actual demand is 0 — common on closed days

    # Log metrics to MLflow for comparison
    mlflow.log_metrics({'rmse': round(p_rmse, 4), 'mape': round(p_mape, 4)})

print(f'✅ Prophet trained and logged to MLflow')
print(f'   RMSE : {p_rmse:.2f} units  (avg error per day in units)')
print(f'   MAPE : {p_mape:.2%}  (avg % error per day)')
print(f'\nINTERVIEW NOTE: MAPE < 10% is considered excellent for retail forecasting.')
print(f'MAPE 10-20% is acceptable. MAPE > 20% needs investigation.')

### 7b · XGBoost

**HOW XGBoost works:**  
XGBoost is an ensemble of decision trees built sequentially. Each tree tries to fix the errors of the previous trees (this is 'boosting'). The key parameters:
- `n_estimators=400` — number of trees. More trees = more complex model, but diminishing returns and slower
- `max_depth=5` — how deep each tree can grow. Deeper = captures more complex patterns but risks overfitting
- `learning_rate=0.05` — how much each tree's predictions count. Lower = slower learning, but usually more accurate
- `subsample=0.85` — each tree sees only 85% of the training rows (reduces overfitting)
- `colsample_bytree=0.80` — each tree sees only 80% of the features (reduces overfitting)
- `reg_alpha` and `reg_lambda` — L1 and L2 regularisation to penalise complexity

In [ ]:
# Prepare feature matrices and target vectors
# HOW: X = feature matrix (input), y = target vector (what we predict)
# FEATURE_COLS ensures we use exactly the same features in train and test
X_train = TRAIN[FEATURE_COLS]
y_train = TRAIN['units_sold']
X_test  = TEST[FEATURE_COLS]
y_test  = TEST['units_sold']

xgb_params = {
    'n_estimators'    : 400,          # number of trees to build
    'max_depth'       : 5,            # max depth per tree (5 = moderate complexity)
    'learning_rate'   : 0.05,         # shrinkage rate — low = more trees needed but better generalisation
    'subsample'       : 0.85,         # fraction of training rows used per tree
    'colsample_bytree': 0.80,         # fraction of features used per tree
    'min_child_weight': 3,            # min samples in a leaf — prevents tiny overfitted leaves
    'reg_alpha'       : 0.1,          # L1 regularisation (drives some weights to zero)
    'reg_lambda'      : 1.0,          # L2 regularisation (shrinks all weights toward zero)
    'random_state'    : 42,           # reproducibility
    'objective'       : 'reg:squarederror',  # regression task (not classification)
}

with mlflow.start_run(run_name=f'xgboost_{CONFIG["store"]}_{CONFIG["product"].replace(" ","_")}'):

    mlflow.log_params({**xgb_params, 'model': 'xgboost',
                       'store': CONFIG['store'], 'product': CONFIG['product']})

    xgb_model = xgb.XGBRegressor(**xgb_params)

    # eval_set lets XGBoost monitor test performance during training
    # verbose=False suppresses per-round output (400 rounds would be noisy)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    XGB_PREDS = xgb_model.predict(X_test).clip(0)  # clip to non-negative

    x_rmse = float(np.sqrt(mean_squared_error(ACTUALS, XGB_PREDS)))
    x_mape = float(np.mean(np.abs((ACTUALS - XGB_PREDS) / (ACTUALS + 1e-8))))

    mlflow.log_metrics({'rmse': round(x_rmse, 4), 'mape': round(x_mape, 4)})

    # Feature importance: how much does each feature reduce prediction error?
    # HOW: XGBoost tracks how often each feature is used in splits and how
    # much it reduces the loss. Higher = more important.
    FEATURE_IMPORTANCE = pd.Series(
        xgb_model.feature_importances_, index=FEATURE_COLS
    ).sort_values(ascending=False)

print(f'✅ XGBoost trained and logged to MLflow')
print(f'   RMSE : {x_rmse:.2f} units')
print(f'   MAPE : {x_mape:.2%}')
print(f'\nTop 5 most important features:')
for feat, score in FEATURE_IMPORTANCE.head(5).items():
    print(f'   {feat:<22}: {score:.4f}')
print()
print('INTERVIEW NOTE: If lag_7 or rolling_mean_7 are not in the top 3,')
print('something may be wrong with the feature engineering.')

### 7c · Ensemble

**WHY does ensembling work?**  
Prophet and XGBoost make *different types of errors*. Prophet may overestimate demand on a specific weekday. XGBoost may underestimate during long holiday stretches where its lags don't capture the full effect. When you average two models whose errors are uncorrelated, the errors partially cancel out, and the combined prediction is more accurate than either alone.

This is called **the wisdom of crowds** in statistics — diverse, independent estimators average to something better than any single estimator.

In [ ]:
# Simple weighted average — 50% Prophet + 50% XGBoost
# WHY equal weights as a baseline: without validation data to tune weights,
# equal weighting is the safest default. In production, you'd use a held-out
# validation set to optimise the weights.
ENS_PREDS = (0.5 * PROPHET_PREDS + 0.5 * XGB_PREDS).clip(0)

e_rmse = float(np.sqrt(mean_squared_error(ACTUALS, ENS_PREDS)))
e_mape = float(np.mean(np.abs((ACTUALS - ENS_PREDS) / (ACTUALS + 1e-8))))

with mlflow.start_run(run_name=f'ensemble_{CONFIG["store"]}_{CONFIG["product"].replace(" ","_")}'):
    mlflow.log_params({'model': 'ensemble_50_50',
                       'store': CONFIG['store'], 'product': CONFIG['product']})
    mlflow.log_metrics({'rmse': round(e_rmse, 4), 'mape': round(e_mape, 4)})

# Model comparison table
comparison = pd.DataFrame({
    'Model'  : ['Prophet', 'XGBoost', 'Ensemble (50/50) ⭐'],
    'RMSE'   : [round(p_rmse, 2), round(x_rmse, 2), round(e_rmse, 2)],
    'MAPE'   : [f'{p_mape:.2%}', f'{x_mape:.2%}', f'{e_mape:.2%}'],
    'Verdict': [
        'Strong seasonal capture',
        'Strong lag/feature capture',
        'Best overall'
    ]
}).set_index('Model')

print('=== Model comparison ===')
display(comparison)

# Confirm ensemble beats both individual models
if e_mape < min(p_mape, x_mape):
    improvement = (min(p_mape, x_mape) - e_mape) / min(p_mape, x_mape) * 100
    print(f'\n✅ Ensemble beats best single model by {improvement:.1f}% on MAPE')
else:
    print('\nNote: Ensemble did not outperform best model — consider adjusting weights')

---
## Stage 8 · Evaluation & business impact

**WHY translate MAPE into dollars?**  
A data scientist says *'our MAPE is 8%'*. An executive hears noise.  
A *strategic* data scientist says *'our model is off by ~8 units per day, which costs the business ~$46 in revenue risk daily, or ~$1,400/month'*. That gets attention and budget.

This translation from statistical metric to business impact is what separates candidates who get hired at the senior level from those who don't.

**Metrics explained:**
- **RMSE** (Root Mean Squared Error): average error in units. Penalises large errors more than small ones. Use this when big misses are very costly.
- **MAPE** (Mean Absolute Percentage Error): average % error. Scale-independent — easy to communicate. Use this for executive reporting.
- **Over-forecast cost**: inventory waste — you produced/ordered units you couldn't sell
- **Under-forecast risk**: revenue you couldn't capture — demand existed but supply didn't

In [ ]:
# ── Forecast vs Actuals chart ────────────────────────────────────────────────
# WHY: Visual inspection is as important as metrics. A model can have a good
# MAPE overall but fail badly on promo weeks — the chart reveals this.
# Check: does the model track the actual peaks and troughs?

test_dates = TEST['date'].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dates, y=ACTUALS,
                         mode='lines', name='Actual',
                         line=dict(color='#1A5276', width=2.5)))
fig.add_trace(go.Scatter(x=test_dates, y=PROPHET_PREDS,
                         mode='lines', name='Prophet',
                         line=dict(color='#8E44AD', width=1.5, dash='dot')))
fig.add_trace(go.Scatter(x=test_dates, y=XGB_PREDS,
                         mode='lines', name='XGBoost',
                         line=dict(color='#27AE60', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=test_dates, y=ENS_PREDS,
                         mode='lines', name='Ensemble ⭐',
                         line=dict(color='#E67E22', width=2)))

fig.update_layout(
    title='Forecast vs Actuals — test period (last 60 days)',
    xaxis_title='Date', yaxis_title='Units sold',
    height=380, legend=dict(orientation='h', y=1.12),
    plot_bgcolor='white'
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()
print('Look for: does the ensemble (orange) track actual (blue) most closely?')
print('Watch for: any promo periods where all models under-predict (common failure mode)')

In [ ]:
# ── Feature importance chart ─────────────────────────────────────────────────
# WHY: Shows which features XGBoost learned to rely on most.
# If lag_7 and rolling_mean_7 dominate, our engineering choices were validated.
# If 'month' or 'day_of_month' dominate, the calendar features may be carrying
# the seasonality that Prophet should handle — investigate further.

fi_top = FEATURE_IMPORTANCE.head(12)

fig = go.Figure(go.Bar(
    x=fi_top.values[::-1],    # reverse so highest bar is at the top
    y=fi_top.index[::-1],
    orientation='h',
    marker_color='#1A5276'
))
fig.update_layout(
    title='XGBoost — top 12 feature importances (gain)',
    xaxis_title='Importance score', height=380,
    plot_bgcolor='white'
)
fig.show()
print(f'Top 3 features: {FEATURE_IMPORTANCE.head(3).index.tolist()}')
print('INTERVIEW TIP: Be ready to explain WHY lag_7 is so important — same-day-last-week captures the weekly cycle better than any engineered feature can.')

In [ ]:
# ── Business dollar impact ────────────────────────────────────────────────────
# HOW: Convert raw prediction errors into business-interpretable numbers

price   = CONFIG['avg_price']
errors  = np.abs(ACTUALS - ENS_PREDS)   # absolute error per day (in units)

# Over-forecast: model predicted MORE than actuals — we over-produced/over-ordered
# np.clip(x, 0, None) keeps only positive values (zeros out negatives)
over_fc  = np.sum(np.clip(ENS_PREDS - ACTUALS, 0, None))

# Under-forecast: model predicted LESS than actuals — we ran out of stock
under_fc = np.sum(np.clip(ACTUALS   - ENS_PREDS, 0, None))

print('=' * 58)
print(f'  BUSINESS IMPACT SUMMARY — ENSEMBLE MODEL')
print(f'  {CONFIG["store"]} | {CONFIG["product"]} | {CONFIG["test_days"]}-day test period')
print('=' * 58)
print(f'  Statistical metrics:')
print(f'    RMSE                     : {e_rmse:.2f} units/day')
print(f'    MAPE                     : {e_mape:.2%}')
print()
print(f'  Operational impact:')
print(f'    Avg daily error          : {errors.mean():.1f} units  (${errors.mean() * price:.2f}/day)')
print(f'    Total over-forecast      : {over_fc:,.0f} units  (${over_fc * price:,.2f} inventory waste)')
print(f'    Total under-forecast     : {under_fc:,.0f} units  (${under_fc * price:,.2f} revenue at risk)')
print()
print(f'  Executive framing:')
print(f'    A 1% MAPE improvement =  ~${under_fc * price * 0.01:,.2f} additional revenue protected')
print(f'    Monthly revenue impact   : ~${errors.mean() * price * 30:,.2f}')
print('=' * 58)
print()
print('INTERVIEW TIP: This is the table you show non-technical stakeholders.')
print('Never show raw RMSE to an executive. Always translate to dollars or % of revenue.')

---
## Stage 9 · Forward forecast (next 30–90 days)

**WHY is forward forecasting different from test-set prediction?**  
On the test set, we already have the actual feature values (including lags) — we just didn't use them to train. For a *true future* forecast, we don't yet have the lag values for the next 30 days because those dates haven't happened yet.

**HOW we handle this:**  
We fill future lag values from the tail of the known series:
- `lag_7` for day `t+1` = actual demand from day `t-6` (7 days before t+1, which we know)
- `lag_7` for day `t+8` = actual demand from day `t+1`... which we just predicted

For simplicity, this pipeline fills all future lags from known history (not iterative prediction). Iterative prediction (using each prediction to fill the next lag) is more accurate but more complex — a good enhancement to mention in interviews.

**WHY ±15% uncertainty bounds?**  
Every forecast has uncertainty that grows with the horizon — we're less sure about day 30 than day 1. The ±15% bounds communicate this uncertainty to stakeholders. In a production system, you'd replace this with *conformal prediction* for statistically valid coverage guarantees — a great thing to mention as a future enhancement.

In [ ]:
def build_future_frame(feat_df, horizon):
    """
    Build a feature DataFrame for the next `horizon` days beyond
    the last date in feat_df.

    HOW: We know the future dates, so we can compute:
    - Calendar features (day_of_week, month, etc.) — these are deterministic
    - is_promo — we know which weeks have promotions in advance
    - is_weekend — deterministic from the date
    - Lag features — filled from the tail of the known series
    - Rolling features — computed from the tail of the known series

    WATCH OUT: We do NOT have future values of units_sold (that's what we're
    predicting!). All features must come from the past or from known future events.
    """
    last_date    = feat_df['date'].max()
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),  # day after the last known date
        periods=horizon,
        freq='D'
    )
    known_vals = feat_df['units_sold'].values  # all historical values we know

    future = pd.DataFrame({'date': future_dates})

    # Calendar features — these are fully deterministic (we know the calendar)
    future['day_of_week']    = future['date'].dt.dayofweek
    future['day_of_month']   = future['date'].dt.day
    future['week_of_year']   = future['date'].dt.isocalendar().week.astype(int)
    future['month']          = future['date'].dt.month
    future['quarter']        = future['date'].dt.quarter
    future['is_month_end']   = future['date'].dt.is_month_end.astype(int)
    future['is_month_start'] = future['date'].dt.is_month_start.astype(int)
    future['is_weekend']     = (future['date'].dt.dayofweek >= 5).astype(int)

    # Known future events
    future['is_holiday'] = 0  # no known holidays in forecast window (conservative)
    future['is_promo']   = future['week_of_year'].isin(PROMO_WEEKS).astype(int)

    # Trend index — continues from where training left off
    future['t'] = np.arange(len(feat_df) + 1, len(feat_df) + 1 + horizon)

    # Lag features — pulled from the known historical tail
    # HOW: for forecast day i (0-indexed), lag_7 = known_vals[-(7) + i]
    # i.e., we reach back 7 days from each forecast date into known history
    for lag in [7, 14, 28]:
        lag_values = []
        for i in range(horizon):
            idx = len(known_vals) - lag + i  # index into historical array
            lag_values.append(known_vals[max(idx, 0)])  # max(idx,0) prevents negative indexing
        future[f'lag_{lag}'] = lag_values

    # Rolling features — computed from the known tail
    # WHY use a fixed tail? For a short horizon, the tail mean is a good
    # approximation of the recent trend entering the forecast window.
    tail_7  = known_vals[-7:]
    tail_28 = known_vals[-28:]
    future['rolling_mean_7']  = np.mean(tail_7)
    future['rolling_std_7']   = np.std(tail_7)
    future['rolling_mean_28'] = np.mean(tail_28)
    future['rolling_std_28']  = np.std(tail_28)
    future['rolling_max_7']   = np.max(tail_7)

    # Price proxy — use the recent 30-day average
    recent = feat_df.tail(30)
    total_rev   = recent['revenue'].sum()
    total_units = recent['units_sold'].replace(0, np.nan).sum()
    future['avg_price'] = (
        (total_rev / total_units) if total_units > 0 else CONFIG['avg_price']
    )

    return future


# Build future features and generate predictions
FUTURE        = build_future_frame(FEAT_DF, CONFIG['horizon'])
FUTURE_PREDS  = xgb_model.predict(FUTURE[FEATURE_COLS]).clip(0)  # XGBoost inference
uncertainty   = FUTURE_PREDS * 0.15   # ±15% uncertainty band

# ── Forward forecast chart ───────────────────────────────────────────────────
tail_hist = FEAT_DF.tail(60)  # show last 60 days of history for context

fig = go.Figure()

# Historical tail — so the forecast doesn't start in a vacuum
fig.add_trace(go.Scatter(
    x=tail_hist['date'], y=tail_hist['units_sold'],
    mode='lines', name='Historical (last 60d)',
    line=dict(color='#1A5276', width=1.5)
))

# Uncertainty band — filled area between upper and lower bounds
# HOW: Plotly fills between two traces. We concatenate forward dates + reversed
# forward dates to create a closed polygon for the fill.
fig.add_trace(go.Scatter(
    x=pd.concat([FUTURE['date'], FUTURE['date'].iloc[::-1]]),
    y=np.concatenate([
        (FUTURE_PREDS + uncertainty),          # upper bound (forward)
        (FUTURE_PREDS - uncertainty)[::-1]     # lower bound (reversed)
    ]),
    fill='toself',
    fillcolor='rgba(230, 126, 34, 0.15)',  # semi-transparent orange
    line=dict(color='rgba(255,255,255,0)'),  # invisible border
    name='±15% uncertainty'
))

# Main forecast line
fig.add_trace(go.Scatter(
    x=FUTURE['date'], y=FUTURE_PREDS.round(0),
    mode='lines+markers', name=f'{CONFIG["horizon"]}-day forecast',
    line=dict(color='#E67E22', width=2),
    marker=dict(size=4)
))

# Star markers on promo days in the forecast window
# WHY: Stakeholders want to see WHERE the model expects promo-driven spikes
promo_mask = FUTURE['is_promo'] == 1
if promo_mask.any():
    fig.add_trace(go.Scatter(
        x=FUTURE[promo_mask]['date'],
        y=FUTURE_PREDS[promo_mask.values],
        mode='markers', name='Promo day',
        marker=dict(color='#F39C12', size=10, symbol='star')
    ))

fig.update_layout(
    title=f'{CONFIG["horizon"]}-day forward forecast — {CONFIG["store"]} | {CONFIG["product"]}',
    xaxis_title='Date', yaxis_title='Predicted units sold',
    height=400, legend=dict(orientation='h', y=1.12),
    plot_bgcolor='white'
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

# Forecast summary statistics
print(f'Forecast summary ({CONFIG["horizon"]} days from {FUTURE["date"].min().date()}):')
print(f'  Avg predicted daily demand : {FUTURE_PREDS.mean():.1f} units')
print(f'  Peak day predicted         : {FUTURE["date"].iloc[FUTURE_PREDS.argmax()].date()} ({FUTURE_PREDS.max():.0f} units)')
print(f'  Promo days in this window  : {promo_mask.sum()}')
print(f'  Projected period revenue   : ${(FUTURE_PREDS * CONFIG["avg_price"]).sum():,.2f}')

---
## Stage 10 · MLflow experiment summary

**WHY MLflow matters for a portfolio:**  
Most candidates build models in notebooks with no tracking. Having MLflow in your project shows you understand:
1. **Reproducibility** — can you recreate the exact run that produced a given metric?
2. **Auditability** — what parameters produced the best model? Which were tried and failed?
3. **Production mindset** — in real organisations, data scientists don't work alone. MLflow lets teams compare each other's experiments.

In a local environment, run `mlflow ui` in the project directory to open the visual experiment browser at `http://localhost:5000`.

In [ ]:
# Retrieve all logged runs from this experiment
runs = mlflow.search_runs(experiment_names=['retail-demand-forecasting'])

if not runs.empty:
    cols_to_show = ['tags.mlflow.runName', 'metrics.rmse', 'metrics.mape']
    available   = [c for c in cols_to_show if c in runs.columns]

    summary = (
        runs[available]
        .rename(columns={
            'tags.mlflow.runName': 'Run name',
            'metrics.rmse'       : 'RMSE',
            'metrics.mape'       : 'MAPE',
        })
        .dropna(subset=['RMSE'])
        .sort_values('RMSE')
    )
    summary['MAPE'] = summary['MAPE'].apply(lambda x: f'{x:.2%}' if pd.notna(x) else '-')
    summary['RMSE'] = summary['RMSE'].round(3)

    print('=== All MLflow experiment runs (sorted by RMSE) ===')
    display(summary)
    print(f'\nTotal runs logged: {len(summary)}')
    print('In a local environment: run `mlflow ui` to view the full visual experiment browser')
else:
    print('No runs found — run the training cells above first')

---
## Stage 11 · Full pipeline summary & next steps

In [ ]:
print('=' * 62)
print('  RETAIL DEMAND FORECASTING PIPELINE — COMPLETE')
print('=' * 62)
print()
print(f'  Configuration')
print(f'  ├─ Store            : {CONFIG["store"]}')
print(f'  ├─ Product          : {CONFIG["product"]}')
print(f'  ├─ Training period  : {TRAIN["date"].min().date()} → {TRAIN["date"].max().date()}')
print(f'  └─ Test period      : {TEST["date"].min().date()} → {TEST["date"].max().date()}')
print()
print(f'  Features engineered : {len(FEATURE_COLS)}')
print(f'  Top feature         : {FEATURE_IMPORTANCE.index[0]} (importance: {FEATURE_IMPORTANCE.iloc[0]:.4f})')
print()
print(f'  Model performance (test set)')
print(f'  ├─ Prophet   RMSE: {p_rmse:>7.2f} units   MAPE: {p_mape:.2%}')
print(f'  ├─ XGBoost   RMSE: {x_rmse:>7.2f} units   MAPE: {x_mape:.2%}')
print(f'  └─ Ensemble  RMSE: {e_rmse:>7.2f} units   MAPE: {e_mape:.2%}  ← best')
print()
print(f'  Business impact (ensemble, test period)')
print(f'  ├─ Avg daily error       : {errors.mean():.1f} units  (${errors.mean() * price:.2f}/day)')
print(f'  ├─ Over-forecast waste   : ${over_fc  * price:,.2f}')
print(f'  └─ Revenue at risk       : ${under_fc * price:,.2f}')
print()
print(f'  Forward forecast')
print(f'  ├─ Horizon              : {CONFIG["horizon"]} days')
print(f'  ├─ Avg predicted demand : {FUTURE_PREDS.mean():.1f} units/day')
print(f'  └─ Projected revenue    : ${(FUTURE_PREDS * price).sum():,.2f}')
print()
print('  Recommended next enhancements')
print('  ├─ Conformal prediction  → statistically valid uncertainty bounds')
print('  ├─ Optuna tuning         → optimise XGBoost hyperparameters automatically')
print('  ├─ Iterative forecasting → use predictions to fill future lags')
print('  ├─ PySpark / Databricks  → scale to all 25 store-product series in parallel')
print('  └─ FastAPI endpoint      → serve predictions via REST API in real time')
print('=' * 62)